## Function 6: Spatial Joins 🔗

**Welcome to your next GeoPandas function!**

In this notebook, you'll learn how to build the `spatial_joins()` function step by step. This is where you combine attributes from different spatial datasets based on location rather than a shared key field.

### 🎯 What This Function Does
- Perform inner, left, and right spatial joins
- Use spatial predicates for joins
- Combine attributes from multiple datasets
- Handle CRS compatibility in joins

### 🔧 Function Signature
```python
def spatial_joins(left_gdf, right_gdf, how='inner', predicate='intersects'):
    """
    Args:
        left_gdf (geopandas.GeoDataFrame): Left spatial dataset
        right_gdf (geopandas.GeoDataFrame): Right spatial dataset
        how (str): Type of join
        predicate (str): Spatial predicate used for matching
    
    Returns:
        geopandas.GeoDataFrame: Joined spatial dataset
    """
```

### 📍 Where This Function Goes:
**Target File**: `src/spatial_basics.py`  
**Function Name**: `spatial_joins()`  
**Replace**: The placeholder function with your working code

---

### 📚 Step 1: Load Data for Spatial Joins

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# Load datasets
cities = gpd.read_file('../data/cities/ne_cities_us.geojson')
ecoregions = gpd.read_file('../data/ecoregions/epa_level3_western_us.geojson')

print(f"✅ Loaded:")
print(f"   Cities: {len(cities)} features")
print(f"   Ecoregions: {len(ecoregions)} features")

# Basic spatial join example
joined = gpd.sjoin(cities, ecoregions, how='left', predicate='within')
print(f"\nBasic spatial join:")
print(f"   Joined features: {len(joined)}")
print(f"   New columns from ecoregions: {[c for c in joined.columns if c not in cities.columns and c != 'index_right'][:3]}...")

---

### 🗺️ Step 2: Visualization 1 - Join Result Before vs After

**Level 3 Goal:** See how spatial joins integrate attributes from multiple datasets


In [ ]:
# Perform left join to keep all cities
cities_with_eco = gpd.sjoin(cities, ecoregions, how='left', predicate='within')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# LEFT: Before join (separate datasets)
ecoregions.plot(ax=ax1, alpha=0.3, edgecolor='black', color='lightgray', linewidth=1, label='Ecoregions')
cities.plot(ax=ax1, color='red', markersize=40, alpha=0.7, label='Cities', edgecolor='white', linewidth=0.5)
ax1.legend(loc='upper left', fontsize=11, framealpha=0.9)
ax1.set_title("Before Join: Separate Datasets\nCities + Ecoregions", fontsize=13, fontweight='bold')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.grid(True, alpha=0.3)

# Add annotation
ax1.text(0.5, 0.02, "Two independent datasets\nNo shared attributes",
        transform=ax1.transAxes,
        ha='center', va='bottom',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8),
        fontsize=10)

# RIGHT: After join (cities colored by ecoregion)
ecoregions.plot(ax=ax2, alpha=0.2, edgecolor='black', color='lightgray', linewidth=1)

# Color cities by their ecoregion (if they have one)
colors = plt.cm.Set3(np.linspace(0, 1, len(ecoregions)))
eco_names = ecoregions['eco_name'].unique() if 'eco_name' in ecoregions.columns else range(len(ecoregions))

for idx, eco_name in enumerate(eco_names):
    if 'eco_name' in cities_with_eco.columns:
        cities_in_eco = cities_with_eco[cities_with_eco['eco_name'] == eco_name]
    else:
        cities_in_eco = cities_with_eco.iloc[0:0]  # Empty
        
    if len(cities_in_eco) > 0:
        label = eco_name[:20] if isinstance(eco_name, str) else f"Eco {eco_name}"
        cities_in_eco.plot(ax=ax2, color=colors[idx], markersize=50, label=label, alpha=0.8, edgecolor='black', linewidth=0.5)

# Cities without ecoregion
if 'eco_name' in cities_with_eco.columns:
    cities_no_eco = cities_with_eco[cities_with_eco['eco_name'].isna()]
    if len(cities_no_eco) > 0:
        cities_no_eco.plot(ax=ax2, color='gray', markersize=50, label='No ecoregion', alpha=0.6, marker='x')

ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, framealpha=0.9)
ax2.set_title("After Join: Cities Inherit Ecoregion Attributes\nColored by Ecoregion", fontsize=13, fontweight='bold')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.grid(True, alpha=0.3)

# Add annotation
matched = (cities_with_eco['eco_name'].notna()).sum() if 'eco_name' in cities_with_eco.columns else 0
ax2.text(0.5, 0.02, f"Joined dataset\n{matched} cities with ecoregion attributes",
        transform=ax2.transAxes,
        ha='center', va='bottom',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),
        fontsize=10)

plt.tight_layout()
plt.show()

print(f"🎯 Spatial Join Result:")
print(f"   Before: {len(cities)} cities (no ecoregion info)")
print(f"   After: {len(cities_with_eco)} cities (WITH ecoregion attributes)")
if 'eco_name' in cities_with_eco.columns:
    matched = (cities_with_eco['eco_name'].notna()).sum()
    print(f"   Matched: {matched} cities now know their ecoregion!")
    print(f"   Unmatched: {len(cities_with_eco) - matched} cities outside ecoregions")


ompare 

### 🗺️ Step 3: Visualization 2 - Compare Join Types

In [ ]:
# Perform all three join types
inner_join = gpd.sjoin(cities, ecoregions, how='inner', predicate='within')
left_join = gpd.sjoin(cities, ecoregions, how='left', predicate='within')
#right_join = gpd.sjoin(cities, ecoregions, how='right', predicate='within')  # Less common

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

# ORIGINAL DATA
ecoregions.plot(ax=ax1, alpha=0.3, edgecolor='black', linewidth=1, color='lightblue')
cities.plot(ax=ax1, color='gray', markersize=30, alpha=0.6)
ax1.set_title(f"Original Data\n{len(cities)} cities, {len(ecoregions)} ecoregions", 
             fontsize=12, fontweight='bold')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.grid(True, alpha=0.3)

# INNER JOIN (only matching)
ecoregions.plot(ax=ax2, alpha=0.3, edgecolor='black', linewidth=1, color='lightblue')
cities.plot(ax=ax2, color='lightgray', markersize=30, alpha=0.3)
inner_join.plot(ax=ax2, color='green', markersize=50, alpha=0.8, edgecolor='darkgreen', linewidth=0.5)
ax2.set_title(f"Inner Join\n{len(inner_join)} cities (only matched)", 
             fontsize=12, fontweight='bold')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.grid(True, alpha=0.3)
ax2.text(0.5, 0.02, "✅ Only cities inside ecoregions",
        transform=ax2.transAxes,
        ha='center', va='bottom',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),
        fontsize=9)

# LEFT JOIN (all from left)
ecoregions.plot(ax=ax3, alpha=0.3, edgecolor='black', linewidth=1, color='lightblue')

if 'eco_name' in left_join.columns:
    matched = left_join[left_join['eco_name'].notna()]
    unmatched = left_join[left_join['eco_name'].isna()]
    
    matched.plot(ax=ax3, color='green', markersize=50, alpha=0.8, label=f'Matched ({len(matched)})', 
                edgecolor='darkgreen', linewidth=0.5)
    if len(unmatched) > 0:
        unmatched.plot(ax=ax3, color='red', markersize=50, alpha=0.8, label=f'Unmatched ({len(unmatched)})', 
                      marker='x', linewidth=2)
    ax3.legend(loc='upper left', fontsize=9)

matched_count = len(matched) if 'eco_name' in left_join.columns and len(left_join) > 0 else 0
unmatched_count = len(unmatched) if 'eco_name' in left_join.columns and len(left_join) > 0 else 0
ax3.set_title(f"Left Join\nAll {len(left_join)} cities\n({matched_count} matched, {unmatched_count} unmatched)", 
             fontsize=12, fontweight='bold')
ax3.set_xlabel('Longitude')
ax3.set_ylabel('Latitude')
ax3.grid(True, alpha=0.3)
ax3.text(0.5, 0.02, "✅ All cities kept\nMatched + Unmatched",
        transform=ax3.transAxes,
        ha='center', va='bottom',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8),
        fontsize=9)

plt.suptitle("Spatial Join Types Comparison", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"🎯 Join Type Comparison:")
print(f"   Original cities: {len(cities)}")
print(f"   Inner join: {len(inner_join)} (only matched)")
print(f"   Left join: {len(left_join)} (all cities, matched + unmatched)")
if 'eco_name' in left_join.columns:
    print(f"   Matched: {(left_join['eco_name'].notna()).sum()}")
    print(f"   Unmatched: {(left_join['eco_name'].isna()).sum()}")


 at 

### 🗺️ Step 4: Visualization 3 - Attribute Summary After Join

In [ ]:
# Use the joined data to calculate statistics by ecoregion
if 'eco_name' in left_join.columns:
    eco_counts = left_join[left_join['eco_name'].notna()].groupby('eco_name').size().sort_values(ascending=False)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    
    # LEFT: Map with count labels
    ecoregions_with_counts = ecoregions.copy()
    ecoregions_with_counts['city_count'] = ecoregions['eco_name'].map(eco_counts).fillna(0)
    
    ecoregions_with_counts.plot(ax=ax1, column='city_count', cmap='YlOrRd', 
                                edgecolor='black', linewidth=1, legend=True,
                                legend_kwds={'label': 'Cities (from join)', 'shrink': 0.6})
    left_join.plot(ax=ax1, color='darkblue', markersize=20, alpha=0.5)
    
    # Add labels
    for idx, eco in ecoregions.iterrows():
        centroid = eco.geometry.centroid
        eco_name = eco['eco_name']
        count = int(eco_counts.get(eco_name, 0))
        if count > 0:
            ax1.annotate(f"{count}\ncities",
                       xy=(centroid.x, centroid.y),
                       ha='center', va='center',
                       fontsize=9, fontweight='bold',
                       bbox=dict(boxstyle='round,pad=0.4', fc='yellow', alpha=0.8, edgecolor='black'))
    
    ax1.set_title("Join Result: Cities per Ecoregion\n(Calculated from Joined Attributes)", 
                 fontsize=13, fontweight='bold')
    ax1.set_xlabel('Longitude')
    ax1.set_ylabel('Latitude')
    ax1.grid(True, alpha=0.3)
    
    # RIGHT: Bar chart
    eco_names_short = [name[:20] for name in eco_counts.index]
    bars = ax2.barh(eco_names_short, eco_counts.values, color='skyblue', edgecolor='black', linewidth=1)
    ax2.set_xlabel("Number of Cities", fontsize=11)
    ax2.set_title("Cities by Ecoregion\n(From Spatial Join)", fontsize=13, fontweight='bold')
    ax2.invert_yaxis()
    ax2.grid(True, axis='x', alpha=0.3)
    
    # Add value labels
    for bar, count in zip(bars, eco_counts.values):
        width = bar.get_width()
        ax2.text(width, bar.get_y() + bar.get_height()/2,
                f' {int(count)}',
                ha='left', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"🎯 Attribute summary (enabled by spatial join):")
    print(f"   Total cities in ecoregions: {eco_counts.sum()}")
    print(f"   Ecoregions with cities: {len(eco_counts)}")
    print(f"   Most cities in one ecoregion: {eco_counts.max()} ({eco_counts.idxmax()})")
    print(f"\n   💡 This analysis was only possible AFTER the spatial join!")
    print(f"      Cities didn't know their ecoregion before joining")


**🎯 Key Takeaways from Spatial Joins:**
- **Spatial joins** combine datasets based on location (not shared keys)
- **Inner join:** Keep only features that match
- **Left join:** Keep all from left dataset, add right attributes where matched
- **Joined data** enables new analyses (e.g., aggregate stats by region)

**Real-world uses:**
- 🏘️ Add census data to addresses (enrich with demographics)
- 🗳️ Assign polling places to voters (determine which location)
- 🌡️ Link weather stations to climate zones (regional analysis)
- 🚦 Count traffic accidents per neighborhood (spatial aggregation)

**Remember:** Always check CRS compatibility before joining!

---


### 🏗️ Step 5: Building the Complete Function

Now let's put everything together into a reusable function. This is what you will implement in `src/spatial_basics.py`.

## 🎯 Function Signature

```python
def spatial_joins(left_gdf: gpd.GeoDataFrame, right_gdf: gpd.GeoDataFrame, how: str = 'inner', predicate: str = 'intersects') -> gpd.GeoDataFrame:
```

**Returns:** Joined GeoDataFrame with combined attributes

## 💡 Implementation Tips

1. Start with the function signature from `src/spatial_basics.py`
2. Read the docstring carefully - it explains everything
3. Look at the test file to see what's expected
4. Use the examples above as reference
5. Test frequently as you implement

**Need help?** Check the README or ask on the course forum!

In [ ]:
import geopandas as gpd

def spatial_joins(
    left_gdf: gpd.GeoDataFrame,
    right_gdf: gpd.GeoDataFrame,
    how: str = 'inner',
    predicate: str = 'intersects'
) -> gpd.GeoDataFrame:
    """
    Join two GeoDataFrames based on spatial relationships.
    
    Combines attribute data from two spatial datasets based on their
    spatial relationship (e.g., points within polygons, lines intersecting areas).
    
    Args:
        left_gdf: Left GeoDataFrame (preserved geometries)
        right_gdf: Right GeoDataFrame (attributes joined)
        how: Type of join ('inner', 'left', 'right')
        predicate: Spatial predicate ('intersects', 'contains', 'within')
        
    Returns:
        Joined GeoDataFrame with combined attributes
        
    Raises:
        ValueError: If inputs are invalid or CRS incompatible
        
    Example:
        >>> # Join cities with their countries
        >>> result = spatial_joins(cities, countries, how='left', predicate='within')
        >>> print(f"Joined {len(result)} features")
    """
    # Validate inputs
    if not isinstance(left_gdf, gpd.GeoDataFrame) or not isinstance(right_gdf, gpd.GeoDataFrame):
        raise ValueError("Both inputs must be GeoDataFrames")
    
    # Check CRS compatibility
    if left_gdf.crs != right_gdf.crs:
        raise ValueError(
            f"CRS mismatch: left has {left_gdf.crs}, right has {right_gdf.crs}. "
            "Transform to same CRS before joining."
        )
    
    # Validate parameters
    valid_how = ['inner', 'left', 'right']
    if how not in valid_how:
        raise ValueError(f"Invalid 'how' parameter '{how}'. Must be one of: {', '.join(valid_how)}")
    
    valid_predicates = ['intersects', 'contains', 'within']
    if predicate not in valid_predicates:
        raise ValueError(f"Invalid 'predicate' parameter '{predicate}'. Must be one of: {', '.join(valid_predicates)}")
    
    # Perform spatial join
    result = gpd.sjoin(left_gdf, right_gdf, how=how, predicate=predicate)
    
    return result


# Function 7: Overlay and Visualize (2 points)

### ✨ Step 6: Test Your Function

Use these checks before moving on:

- Run the function with **`how='inner'`** and confirm only matching features are kept
- Run the function with **`how='left'`** and confirm all left-side features remain
- Try different predicates such as **`within`** and **`intersects`**
- Confirm joined attributes from the right dataset appear in the output
- Test mismatched CRS and confirm it raises a **ValueError**
- Test an invalid `how` or `predicate` value and confirm it raises a **ValueError**

---

### 🧪 Step 7: Verify with Pytest

After implementing in `../src/spatial_basics.py`:

```bash
uv run pytest tests/ -k "TestSpatialJoins" -v
```

**Check your score:**
```bash
uv run python scripts/calculate_grade.py .
```

**⚠️ IMPORTANT: Make sure this passes before you move on!**

---

### 🔑 Key Learning Points

- **Spatial joins** combine datasets using location rather than a shared attribute key
- **`gpd.sjoin()`** is the main GeoPandas function for joining by location
- **`how='inner'`** keeps only matched features
- **`how='left'`** keeps all features from the left dataset
- **Predicates** like `within`, `contains`, and `intersects` control how matches are found
- CRS compatibility matters before running a spatial join
- Joined outputs make new attribute-based analyses possible